# Demo tool GIS (NDVI) bằng dữ liệu mẫu – Khánh Hòa

Notebook này giúp bạn demo nhanh theo quy trình:
1. Đọc **shapefile .zip** của Khánh Hòa (phường/xã)
2. Đọc **CSV NDVI mẫu** (đã tạo sẵn)
3. Join theo `ma_xa` (khuyến nghị) hoặc `ten_xa`
4. Xuất **CSV hoàn chỉnh** để đưa vào tool demo

> Nếu máy bạn đọc shapefile bằng `geopandas` bị lỗi, notebook có sẵn **fallback** đọc thuộc tính bằng `fiona`.


## 1) Khai báo thư viện

In [ ]:
# Thư viện chính
import os, zipfile, tempfile
import pandas as pd
import numpy as np

# Đọc shapefile (ưu tiên geopandas)
import geopandas as gpd

# Fallback nếu geopandas gặp lỗi
import fiona


## 2) Cấu hình đầu vào/đầu ra

In [ ]:
ZIP_SHP = r"/mnt/data/Khánh Hòa (phường xã) - 34.zip"                 # shapefile zip Khánh Hòa
CSV_NDVI_SAMPLE = r"/mnt/data/ndvi_khanhhoa_sample.csv"              # CSV NDVI mẫu (đã tạo)
OUT_JOINED_CSV = "ndvi_khanhhoa_demo_joined.csv"

# Cột join (khuyến nghị dùng mã để chắc chắn khớp)
JOIN_KEY_SHAPE = "ma_xa"
JOIN_KEY_CSV   = "ma_xa"

# Cột tên hiển thị
NAME_COL = "ten_xa"


## 3) Giải nén shapefile zip

In [ ]:
tmpdir = tempfile.mkdtemp()
with zipfile.ZipFile(ZIP_SHP, 'r') as zf:
    zf.extractall(tmpdir)

shp_path = None
for root, _, files in os.walk(tmpdir):
    for f in files:
        if f.lower().endswith('.shp'):
            shp_path = os.path.join(root, f)
            break
    if shp_path:
        break

print('Extracted shp:', shp_path)


## 4) Đọc shapefile (geopandas) – nếu lỗi sẽ dùng fiona

In [ ]:
try:
    gdf = gpd.read_file(shp_path)
    print('Read with geopandas OK')
    print('Rows:', len(gdf))
    print('Columns:', list(gdf.columns))
except Exception as e:
    print('Geopandas read failed, fallback to fiona...')
    print('Error:', e)

    with fiona.open(shp_path) as src:
        records = [feat['properties'] for feat in src]

    # chỉ có bảng thuộc tính, không có geometry
    gdf = pd.DataFrame.from_records(records)
    print('Rows:', len(gdf))
    print('Columns:', list(gdf.columns))


## 5) Đọc CSV NDVI mẫu

In [ ]:
ndvi_df = pd.read_csv(CSV_NDVI_SAMPLE)
print('NDVI sample rows:', len(ndvi_df))
ndvi_df.head()


## 6) Join shapefile ↔ NDVI mẫu, rồi xuất CSV demo

In [ ]:
# Nếu gdf là GeoDataFrame thì vẫn join bình thường (giữ geometry)
joined = gdf.merge(ndvi_df, left_on=JOIN_KEY_SHAPE, right_on=JOIN_KEY_CSV, how='left')

# Kiểm tra các dòng không match
miss = joined['ndvi_mean'].isna().sum()
print('Missing NDVI rows:', miss)

# Xuất CSV demo (bỏ geometry nếu có)
if hasattr(joined, 'geometry'):
    out_df = pd.DataFrame(joined.drop(columns=['geometry'], errors='ignore'))
else:
    out_df = joined.copy()

out_df.to_csv(OUT_JOINED_CSV, index=False, encoding='utf-8-sig')
print('Saved:', OUT_JOINED_CSV)

out_df[[JOIN_KEY_SHAPE, NAME_COL, 'ndvi_mean', 'ndvi_min', 'ndvi_max', 'ndvi_std', 'count', 'alert_low_green']].head()


## 7) (Tuỳ chọn) Top/Bottom để demo nhanh

In [ ]:
df = out_df.copy()

print('--- Bottom 10 (ít xanh nhất) ---')
display(df.sort_values('ndvi_mean', ascending=True).head(10)[[NAME_COL, 'ndvi_mean', 'alert_low_green']])

print('--- Top 10 (xanh nhất) ---')
display(df.sort_values('ndvi_mean', ascending=False).head(10)[[NAME_COL, 'ndvi_mean', 'alert_low_green']])
